# 02 - Ablation Study

This notebook is reserved for the ablation part of the project.

Scope:
- Table 5: stability under different K0
- Table 6: lambda sensitivity
- Table 7: split and merge component analysis
- Table 8: custom ablation on head initialization


## Optional Regeneration

Current local PnP preset in this repo:
- `epochs=80`
- `warmup_epochs=30`
- `lambda_param=2.0`

The next code cell will regenerate missing ablation CSV files when needed. It also prepares the custom initialization ablation used in the report.


In [ ]:
from pathlib import Path
import subprocess
import pandas as pd

ROOT = Path('..').resolve()
RESULT_DIR = ROOT / 'data' / 'scan_results'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

required_tables = [
    RESULT_DIR / 'table5_k0_stability.csv',
    RESULT_DIR / 'table6_lambda_ablation.csv',
    RESULT_DIR / 'table7_ablation_components.csv',
]

if any(not path.exists() for path in required_tables):
    subprocess.run(
        ['python', 'scripts/generate_reproduction_tables.py', '--tables', 'table5', 'table6', 'table7'],
        check=True,
        cwd=ROOT,
    )

custom_ablation_path = RESULT_DIR / 'table8_init_ablation.csv'
if not custom_ablation_path.exists():
    subprocess.run(['python', 'scripts/generate_extension_tables.py'], check=True, cwd=ROOT)

table5 = pd.read_csv(RESULT_DIR / 'table5_k0_stability.csv')
table6 = pd.read_csv(RESULT_DIR / 'table6_lambda_ablation.csv')
table7 = pd.read_csv(RESULT_DIR / 'table7_ablation_components.csv')
table8 = pd.read_csv(custom_ablation_path)

RESULT_DIR


## Table 5

Robustness with respect to the initial number of clusters.


In [ ]:
display(table5)


## Table 6

Sensitivity of local PnP results to `lambda_param`.


In [ ]:
display(table6)


## Table 7

Component ablation for split, merge, and related controls.


In [ ]:
display(table7)


## Table 8

Custom ablation designed for this project: compare `kmeans` head initialization against `random` initialization under the same local training schedule. This ablation is not copied from the paper and is meant to test whether initialization alone explains the reproduction gap.


In [ ]:
display(table8)

summary = (
    table8.pivot_table(
        index=['Dataset', 'Method', 'K0'],
        columns='Init strategy',
        values=['ACC(%)', 'NMI(%)', 'ARI(%)', 'K*'],
    )
    .sort_index(axis=1)
)
display(summary)


## Analysis

Suggested discussion points for the report:
- Table 5 shows that the local implementation still depends strongly on the starting `K0`. The best local region on CIFAR-10 is around `K0=10` to `K0=15`, while very small or very large `K0` values cause a clear drop.
- Table 6 is unusually flat on the local side. This means the current `lambda_param` sweep is not yet driving materially different fixed points, so the implementation may be dominated by the same clustering state before the threshold changes can matter.
- Table 7 contains several repeated local rows across different ablation settings. That is a sign that some control switches are currently not changing the final assignments enough to produce distinct outcomes on CIFAR-10.
- Table 8 shows that initialization matters, but it is not the whole story. On `SCAN (local)`, random initialization is slightly worse than `kmeans`. On `Ours (local, K0=20)`, random initialization is actually better than `kmeans`, so the remaining deviation from the paper cannot be blamed on initialization alone.
- A more plausible explanation is that the local split and merge schedule, especially the cooldown and threshold logic during cluster-count transitions, is still shaping the optimization path differently from the paper.
